In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/utils/FFT_perso.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8533*2*pi
ω_b = 7.0515*2*pi

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.3


##Drive


ω_d=ω_b# Drive frequency
ω_p=2ω_a-ω_d

ϵ_p = 0.9#4581818181818182
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = κ_b/2#3e-3*κ_2 #from Leon's paper

# First change of variables
Q_a, Q_b = 1/κ_a, 1/κ_b
Q_a_tilde, Q_b_tilde = ω_a/κ_a, ω_b/κ_b
β_a, β_b = 4E_J*sin(ϵ_p)φ_a^2/(ω_a), 4E_J*sin(ϵ_p)φ_b^2/ω_b
λ_a, λ_b = φ_a, φ_b
ϵ_λ_d = ϵ_d*2*λ_b/ω_b
ν_d = 1;

# Second change of variables
κ_a_tilde, κ_b_tilde = 1/(4*Q_a_tilde*sqrt(1-1/(4Q_b_tilde^2))), 1/(2*Q_b_tilde*sqrt(1-1/(4Q_b_tilde^2)))
β_a_tilde, β_b_tilde = β_a/(2-1/(2Q_b_tilde^2)), β_b/(1-1/(4Q_b_tilde^2))
ϵ_d_tilde = ϵ_λ_d/(1-1/(4Q_b_tilde^2))
ν_d_tilde = ν_d/sqrt(1-1/(4Q_b_tilde^2))
γ = (1-1/(4Q_a_tilde^2))/(1-1/(4Q_b_tilde^2))/2;

In [ ]:
#ATS + Mean_Field + ladder_operator
function test_equation!(du,u,p::SimulationParams{SystemATS_ladder_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_p, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    a = u[1]
    b = u[2]

    sin_pump =  - 2*E_J*sin(ϵ_p*cos(ω_p*t))
    a_a_st = real(a*conj(a))
    J_1 = besselj(1, sqrt(2Z_a)*sqrt(a_a_st)/ϕ_quantum)
    J_2 = besselj(2, sqrt(2Z_a)*sqrt(a_a_st)/ϕ_quantum)
    J_3 = besselj(3, sqrt(2Z_a)*sqrt(a_a_st)/ϕ_quantum)
    #coupling_b = -sqrt(Z_b)/sqrt(2)/ϕ_quantum*a^2*J_2/(a_a_st)
    #coupling_a = -sqrt(Z_b)/sqrt(2)/ϕ_quantum*(sqrt(Z_a)/sqrt(2)/ϕ_quantum*(J_1-J_3)/(2*a_a_st^(3/2))*(a^3*conj(b)+a_a_st*conj(a)*b)+J_2*(-a^3*conj(b)/a_a_st^2+b/a))
    coupling_a = Z_a*sqrt(Z_b)*sqrt(2)*(-conj(a)*b+4.5*(a^3*conj(b)+3*a*conj(a)^2*b)/12*Z_a/ϕ_quantum^2)/4/ϕ_quantum^3
    coupling_b = Z_a*sqrt(Z_b)*sqrt(2)*(-a^2+4.5*a^3*conj(a)/6*Z_a/ϕ_quantum^2)/8/ϕ_quantum^3
        
    du[1] = -1im*(ω_a*a + sin_pump*coupling_a) - p.environment_prepared.κ_a * a
    du[2] = -1im*(ω_b*b + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase) + sin_pump*coupling_b) - p.environment_prepared.κ_b * b
    return nothing
end

In [ ]:
t_max=20/κ_b
a_0 = -1-1im
b_0 = 1im

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
drive = DriveTwoModesJA(ϵ_d, ω_d, buffer_phase, ϵ_p, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)
a, b, t_list = simulator(t_max, system, drive, environment; equation = test_equation!);
a_rot = a.*exp.(1im*ω_a*t_list);

In [ ]:
plot(real.(a_rot[1:end]), imag.(a_rot[1:end]), label="")
plot!(xlabel=L"\Re(a)", ylabel=L"\Im(a)")

In [ ]:
t_max = 150/κ_b
t_min = 0.0
t_min_save = t_max - 30/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(ω_a)/sqrt(2)
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)

ϵ_0 = 0.62
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d = 10.12*g2
drive = DriveTwoModesBasics(ϵ_d, 2ω_a, buffer_phase, ϵ_p, 0.0, pump_phase)
drive_prepared = prepare(drive)

t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
Im_a_list=range(-4,4,10)
Re_a_list=range(-4,4,10)

environment = MeanField(κ_a, κ_b)
Final_mode_a = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
Final_mode_b = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
for i in 1:length(Re_a_list)
    re_i = Re_a_list[i]
    for j in 1:length(Im_a_list)
        im_j = Im_a_list[j]
        a_0_j = re_i + 1im*im_j
        b_0_j = a_0_j

        system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0_j, b_0_j)
        system_prepared = prepare(system)
        environment_prepared = prepare(system,environment);

        u_0 = get_init(system_prepared, environment_prepared)
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = test_equation!)

        x_a, p_a = (a+conj(a))/sqrt(2), 1im*(conj(a)-a)/sqrt(2)
        x_b, p_b = (b+conj(b))/sqrt(2), 1im*(conj(b)-b)/sqrt(2)
        x_lam_a, x_lam_b, p_lam_a, p_lam_b = sqrt(2)*λ_a*x_a, sqrt(2)*λ_b*x_b, sqrt(2)*λ_a*p_a, sqrt(2)*λ_b*p_b
        x_a_tilde, x_b_tilde, p_a_tilde, p_b_tilde = x_lam_a, x_lam_b, (p_lam_a+x_lam_a/(2*Q_b_tilde))/sqrt(1-1/(4*Q_b_tilde^2)), (p_lam_b+x_lam_b/(2*Q_b_tilde))/sqrt(1-1/(4*Q_b_tilde^2))
        #s = sqrt(1-1/(4*Q_b_tilde^2))*t_list
        u_a =@. cos(ω_a*t_list)*x_a_tilde - sin(ω_b*t_list/2)*p_a_tilde
        v_a =@. sin(ω_b*t_list/2)*x_a_tilde + cos(ω_b*t_list/2)*p_a_tilde
        u_b =@. cos(ω_b*t_list)*x_b_tilde - sin(ω_b*t_list)*p_b_tilde
        v_b =@. sin(ω_b*t_list)*x_b_tilde + cos(ω_b*t_list)*p_b_tilde;

        a = (x_a_tilde+1im*p_a_tilde)/sqrt(2)
        b = (x_b_tilde+1im*p_b_tilde)/sqrt(2)

        a_rot = a.*exp.(1im*ω_a*t_list)
        b_rot = b.*exp.(1im*2ω_a*t_list)

        Final_mode_b[(re_i,im_j)] = b_rot
        Final_mode_a[(re_i,im_j)] = a_rot
    end
end

In [ ]:
p0 = plot()

# Overlay scatter points
for (re_i, im_j) in keys(Final_mode_a)
    z = Final_mode_a[(re_i, im_j)]
    scatter!(
        p0,
        real.(z),
        imag.(z),
        ms = 1,
        color =:black,
        label = "",
    )
end

# Add labels and title. 
plot!(p0,
        xlabel=L"\Re(a)",
        ylabel=L"\Im(a)",
        title=L"\kappa_a="*string(round(κ_a,digits=6))*" GHz",
        top_margin=5mm)

#savefig("../figures/chaos/poincare_chiral_2_memory_22_01_2026.pdf")

In [ ]:
p0 = plot()

# Overlay scatter points
for (re_i, im_j) in keys(Final_mode_a)
    z = Final_mode_b[(re_i, im_j)]
    scatter!(
        p0,
        real.(z),
        imag.(z),
        ms = 1,
        color =:black,
        label = "",
    )
end

# Add labels and title. 
plot!(p0,
        xlabel=L"\Re(a)",
        ylabel=L"\Im(a)",
        title=L"\kappa_a="*string(round(κ_a,digits=6))*" GHz",
        top_margin=5mm)

#savefig("../figures/chaos/poincare_chiral_2_buffer_22_01_2026.pdf")

In [ ]:
abs(1.55+1im*1.15)

In [ ]:
abs(0.5-0.25*1im)

In [ ]:
(1.9300259065618783/0.5590169943749475)^-1
